In [2]:
pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 40.8 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 59.6 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 77.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 63.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 48.3 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 60.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.0 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 89.4 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 79.0 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 50.7 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [18]:
# =============================
# Préparation des données
# =============================
SPECIAL_TOKENS = {"pad": "<pad>", "sos": "<sos>", "eos": "<eos>", "unk": "<unk>"}

class Vocab:
    def __init__(self):
        self.stoi = {}
        self.itos = []
        for tok in SPECIAL_TOKENS.values():
            self.add(tok)
        self.pad_idx = self.stoi[SPECIAL_TOKENS["pad"]]
        self.sos_idx = self.stoi[SPECIAL_TOKENS["sos"]]
        self.eos_idx = self.stoi[SPECIAL_TOKENS["eos"]]
        self.unk_idx = self.stoi[SPECIAL_TOKENS["unk"]]

    def add(self, token):
        if token not in self.stoi:
            self.stoi[token] = len(self.itos)
            self.itos.append(token)

    def build(self, sentences):
        for sent in sentences:
            for tok in sent:
                self.add(tok)

    def encode(self, sentence):
        return [self.stoi.get(tok, self.unk_idx) for tok in sentence]

    def decode(self, ids):
        return [self.itos[i] for i in ids]

    def __len__(self):
        return len(self.itos)

class TranslationDataset(Dataset):
    def __init__(self, src_file, trg_file, src_vocab, trg_vocab, max_len=50):
        with open(src_file, encoding="utf-8") as f:
            src_lines = [line.strip().lower().split() for line in f]
        with open(trg_file, encoding="utf-8") as f:
            trg_lines = [line.strip().lower().split() for line in f]
        assert len(src_lines) == len(trg_lines)

        src_vocab.build(src_lines)
        trg_vocab.build(trg_lines)

        self.data = []
        for s, t in zip(src_lines, trg_lines):
            if len(s) <= max_len and len(t) <= max_len:
                self.data.append((s, t))
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, trg = self.data[idx]
        src_ids = [self.src_vocab.sos_idx] + self.src_vocab.encode(src) + [self.src_vocab.eos_idx]
        trg_ids = [self.trg_vocab.sos_idx] + self.trg_vocab.encode(trg) + [self.trg_vocab.eos_idx]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_seqs, trg_seqs = zip(*batch)
    src_lens = [len(s) for s in src_seqs]
    trg_lens = [len(t) for t in trg_seqs]
    src_pad = nn.utils.rnn.pad_sequence(src_seqs, padding_value=0)
    trg_pad = nn.utils.rnn.pad_sequence(trg_seqs, padding_value=0)
    return src_pad, trg_pad




In [19]:
# =============================
# Encoder
# =============================
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, rnn_type="gru"):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        if rnn_type == "gru":
            self.rnn = nn.GRU(emb_dim, hid_dim)
        else:
            self.rnn = nn.LSTM(emb_dim, hid_dim)
        self.rnn_type = rnn_type

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return hidden

In [20]:
# =============================
# Decoder
# =============================
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, rnn_type="gru"):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        if rnn_type == "gru":
            self.rnn = nn.GRU(emb_dim, hid_dim)
        else:
            self.rnn = nn.LSTM(emb_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.rnn_type = rnn_type

    def forward(self, input, hidden):
        input = input.unsqueeze(0)  # [1, batch]
        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden

In [12]:
# =============================
# Seq2Seq
# =============================
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        max_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(max_len, batch_size, trg_vocab_size).to(self.device)

        hidden = self.encoder(src)
        input = trg[0, :]  # premier token <sos>

        for t in range(1, max_len):
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if torch.rand(1).item() < teacher_forcing_ratio else top1
        return outputs




In [21]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5, forced_teacher=False):
        batch_size = trg.shape[1]
        max_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(max_len, batch_size, trg_vocab_size).to(self.device)

        hidden = self.encoder(src)
        input = trg[0, :]  # premier token <sos>

        for t in range(1, max_len):
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output

            # take best prediction
            top1 = output.argmax(1)

            if forced_teacher:
                # always ground truth
                input = trg[t]
            else:
                # mix between GT and prediction
                teacher_force = torch.rand(1).item() < teacher_forcing_ratio
                input = trg[t] if teacher_force else top1

        return outputs


In [7]:
# =============================
# Exemple jouet
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM = 50   # taille vocab source
OUTPUT_DIM = 50  # taille vocab cible
EMB_DIM = 32
HID_DIM = 64

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
model = Seq2Seq(enc, dec, device).to(device)

# Données jouet : (longueur, batch)
src = torch.randint(0, INPUT_DIM, (10, 32)).to(device)
trg = torch.randint(0, OUTPUT_DIM, (12, 32)).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(5):
    optimizer.zero_grad()
    output = model(src, trg)
    output_dim = output.shape[-1]
    output = output[1:].view(-1, output_dim)
    trg_y = trg[1:].view(-1)
    loss = criterion(output, trg_y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss = {loss.item():.3f}")

Epoch 1, Loss = 3.927
Epoch 2, Loss = 3.922
Epoch 3, Loss = 3.918
Epoch 4, Loss = 3.901
Epoch 5, Loss = 3.893


In [13]:
# =============================
# Exemple d’entraînement
# =============================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    src_vocab, trg_vocab = Vocab(), Vocab()
    train_data = TranslationDataset("small_vocab_fr.txt", "small_vocab_en.txt", src_vocab, trg_vocab)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)

    INPUT_DIM = len(src_vocab)
    OUTPUT_DIM = len(trg_vocab)
    EMB_DIM = 64
    HID_DIM = 128

    enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    model = Seq2Seq(enc, dec, device).to(device)

    optimizer = optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss(ignore_index=src_vocab.pad_idx)

    for epoch in range(5):
        total_loss = 0
        for src, trg in train_loader:
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad()
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_y = trg[1:].view(-1)
            loss = criterion(output, trg_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss={total_loss/len(train_loader):.3f}")

KeyboardInterrupt: 

In [22]:
from tqdm import tqdm

# =============================
# Exemple d’entraînement
# =============================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    src_vocab, trg_vocab = Vocab(), Vocab()
    train_data = TranslationDataset("small_vocab_fr.txt", "small_vocab_en.txt", src_vocab, trg_vocab)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)

    INPUT_DIM = len(src_vocab)
    OUTPUT_DIM = len(trg_vocab)
    EMB_DIM = 64
    HID_DIM = 128

    enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    model = Seq2Seq(enc, dec, device).to(device)

    optimizer = optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss(ignore_index=src_vocab.pad_idx)

    for epoch in range(5):
        total_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
        for src, trg in loop:
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad()
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_y = trg[1:].view(-1)
            loss = criterion(output, trg_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        print(f"Epoch {epoch+1}, Loss={total_loss/len(train_loader):.3f}")


Epoch 1, Loss=0.910


Epoch 2, Loss=0.338


Epoch 3, Loss=0.199


Epoch 4, Loss=0.163


Epoch 5, Loss=0.153


In [14]:
from tqdm import tqdm

# =============================
# Exemple d’entraînement
# =============================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    src_vocab, trg_vocab = Vocab(), Vocab()
    train_data = TranslationDataset("small_vocab_fr.txt", "small_vocab_en.txt", src_vocab, trg_vocab)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)

    INPUT_DIM = len(src_vocab)
    OUTPUT_DIM = len(trg_vocab)
    EMB_DIM = 64
    HID_DIM = 128

    enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
    model = Seq2Seq(enc, dec, device).to(device)

    optimizer = optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss(ignore_index=src_vocab.pad_idx)

    for epoch in range(5):
        total_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
        for src, trg in loop:
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad()
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_y = trg[1:].view(-1)
            loss = criterion(output, trg_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        print(f"Epoch {epoch+1}, Loss={total_loss/len(train_loader):.3f}")


Epoch 1, Loss=0.820


Epoch 2, Loss=0.264


Epoch 3, Loss=0.195


Epoch 4, Loss=0.176


Epoch 5, Loss=0.152


In [15]:
# After training
torch.save(model.state_dict(), "seq2seq_model.pt")


In [ ]:
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, rnn_type="gru")
model = Seq2Seq(enc, dec, device).to(device)

model.load_state_dict(torch.load("seq2seq_model.pt", map_location=device))
model.eval()


In [16]:
torch.save(model, "seq2seq_full.pt")


In [ ]:
model = torch.load("seq2seq_full.pt", map_location=device)
model.eval()


In [ ]:
# After training
torch.save(model.state_dict(), "seq2seq_model_ft.pt")


In [ ]:
torch.save(model, "seq2seq_full_ft.pt")
